# Data Cleaning

This notebook cleans the raw dataset based on findings from `01_exploration.ipynb`.


## 1. Setup & Load Raw Data


In [8]:
# import and load data
import pandas as pd

data = pd.read_csv("../data/kaggle_data/simpsons_episodes.csv")
print(f"Raw data: {data.shape[0]} rows, {data.shape[1]} columns")
data.head()

Raw data: 600 rows, 14 columns


,id,image_url,imdb_rating,imdb_votes,number_in_season,number_in_series,original_air_date,original_air_year,production_code,season,title,us_viewers_in_millions,video_url,views
0,10,http://static-media.fxx.com/img/FX_Networks_-_...,7.4,1511.0,10,10,1990-03-25,1990,7G10,1,Homer's Night Out,30.3,http://www.simpsonsworld.com/video/275197507879,50816.0
1,12,http://static-media.fxx.com/img/FX_Networks_-_...,8.3,1716.0,12,12,1990-04-29,1990,7G12,1,Krusty Gets Busted,30.4,http://www.simpsonsworld.com/video/288019523914,62561.0
2,14,http://static-media.fxx.com/img/FX_Networks_-_...,8.2,1638.0,1,14,1990-10-11,1990,7F03,2,"Bart Gets an ""F""",33.6,http://www.simpsonsworld.com/video/260539459671,59575.0
3,17,http://static-media.fxx.com/img/FX_Networks_-_...,8.1,1457.0,4,17,1990-11-01,1990,7F01,2,Two Cars in Every Garage and Three Eyes on Eve...,26.1,http://www.simpsonsworld.com/video/260537411822,64959.0
4,19,http://static-media.fxx.com/img/FX_Networks_-_...,8.0,1366.0,6,19,1990-11-15,1990,7F08,2,Dead Putting Society,25.4,http://www.simpsonsworld.com/video/260539459670,50691.0


## 2. Drop Season 28

Season 28 only has 4 of 22 episodes in the dataset, with 3 of them missing ratings and viewership entirely. We therefore decided to exclude this season from our analysis.


In [9]:
data = data[data["season"] != 28].copy()
print(f"After dropping season 28: {data.shape[0]} rows")
print(f"Seasons: {data['season'].min()} - {data['season'].max()}")

After dropping season 28: 596 rows
Seasons: 1 - 27


## 3. Fill Missing Viewer Counts (Season 8)

Three season 8 episodes are missing `us_viewers_in_millions` but have all other data intact. Values looked up manually from Wikipedia: https://en.wikipedia.org/wiki/List_of_The_Simpsons_episodes_(seasons_1%E2%80%9320)


In [ ]:
# fix missing values in viewers column for specific episodes
# source:
viewer_fixes = {
    160: 12.2,  # Lisa's Date with Density (S08E07)
    161: 14.36,  # Hurricane Neddy (S08E08)
    173: 13.25,  # The Canine Mutiny (S08E20)
}

for ep_num, viewers in viewer_fixes.items():
    data.loc[data["number_in_series"] == ep_num, "us_viewers_in_millions"] = viewers

# Verify no missing values remain in key columns
print("Remaining nulls in key columns:")
print(data[["imdb_rating", "imdb_votes", "us_viewers_in_millions"]].isnull().sum())

Remaining nulls in key columns:
imdb_rating               0
imdb_votes                0
us_viewers_in_millions    0
dtype: int64


## 4. Parse Dates & Derive Weekday

`original_air_date` is stored as a string. Converting to datetime allows us to derive the day of the week, which we need for the weekday-viewership analysis.


In [11]:
data["original_air_date"] = pd.to_datetime(data["original_air_date"], format="%Y-%m-%d")
data["weekday"] = data["original_air_date"].dt.day_name()

print(
    f"Date range: {data['original_air_date'].min()} to {data['original_air_date'].max()}"
)
print(f"\nWeekday distribution:")
print(data["weekday"].value_counts())

Date range: 1989-12-17 00:00:00 to 2016-05-22 00:00:00

Weekday distribution:
weekday
Sunday       502
Thursday      89
Tuesday        2
Wednesday      2
Friday         1
Name: count, dtype: int64


## 5. Drop Redundant / Unused Columns

- `id`: identical to `number_in_series`
- `original_air_year`: derivable from `original_air_date`
- `image_url`, `video_url`: URLs, not needed for analysis
- `production_code`: internal Fox code, no analytical value
- `views`: likely website page views, not TV audience


In [12]:
cols_to_drop = [
    "id",
    "original_air_year",
    "image_url",
    "video_url",
    "production_code",
    "views",
]
data = data.drop(columns=cols_to_drop)

print(f"Remaining columns: {list(data.columns)}")

Remaining columns: ['imdb_rating', 'imdb_votes', 'number_in_season', 'number_in_series', 'original_air_date', 'season', 'title', 'us_viewers_in_millions', 'weekday']


## 6. Sort & Reset Index


In [ ]:
data = data.sort_values("number_in_series").reset_index(drop=True)

# Reorder columns: identification, temporal, metrics
data = data[
    [
        "title",
        "season",
        "number_in_season",
        "number_in_series",
        "original_air_date",
        "weekday",
        "imdb_rating",
        "imdb_votes",
        "us_viewers_in_millions",
    ]
]

print(
    f"Sorted. First episode: {data.iloc[0]['title']} (S{int(data.iloc[0]['season']):02d}E{int(data.iloc[0]['number_in_season']):02d})"
)
print(
    f"Last episode:  {data.iloc[-1]['title']} (S{int(data.iloc[-1]['season']):02d}E{int(data.iloc[-1]['number_in_season']):02d})"
)

Sorted. First episode: Simpsons Roasting on an Open Fire (S01E01)
Last episode:  Orange Is the New Yellow (S27E22)


## 7. Final Verification


In [14]:
print(f"Shape: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"Seasons: {data['season'].min()} to {data['season'].max()}")
print(f"\nNull counts:")
print(data.isnull().sum())
print(f"\nDtypes:")
print(data.dtypes)
data.head()

Shape: 596 rows, 9 columns
Seasons: 1 to 27

Null counts:
title                     0
season                    0
number_in_season          0
number_in_series          0
original_air_date         0
weekday                   0
imdb_rating               0
imdb_votes                0
us_viewers_in_millions    0
dtype: int64

Dtypes:
title                                str
season                             int64
number_in_season                   int64
number_in_series                   int64
original_air_date         datetime64[us]
weekday                              str
imdb_rating                      float64
imdb_votes                       float64
us_viewers_in_millions           float64
dtype: object


,title,season,number_in_season,number_in_series,original_air_date,weekday,imdb_rating,imdb_votes,us_viewers_in_millions
0,Simpsons Roasting on an Open Fire,1,1,1,1989-12-17,Sunday,8.2,3734.0,26.7
1,Bart the Genius,1,2,2,1990-01-14,Sunday,7.8,1973.0,24.5
2,Homer's Odyssey,1,3,3,1990-01-21,Sunday,7.5,1709.0,27.5
3,There's No Disgrace Like Home,1,4,4,1990-01-28,Sunday,7.8,1701.0,20.2
4,Bart the General,1,5,5,1990-02-04,Sunday,8.1,1732.0,27.1


## 8. Save Clean Dataset


In [15]:
data.to_csv("../data/clean_data/simpsons_episodes_clean.csv", index=False)
print("Saved to data/clean_data/simpsons_episodes_clean.csv")

Saved to data/clean_data/simpsons_episodes_clean.csv
